In [74]:
import re
import random
import argparse

from pathlib import Path
from datetime import datetime
from typing import Any, List, Tuple

import numpy as np
import torch as th
import torch.nn as nn
import gymnasium as gym

from torch.optim import Adam
from torch.nested import nested_tensor

from carl.gymnasium import CARLTorchVectorEnv
from carl.gymnasium import CARLObservation
from jarl.collect import CriticCapture, LogProbCapture, SelfPlayMatchmaker, SelfPlayRunner
from jarl.data.batch import TensorBatch
from jarl.learn import (
    Algorithm,
    IndependentOptimizerSteps,
    OptimizerStep,
    PPOConfig,
    PPOLoss,
    Update,
)
from jarl.log.logger import Logger
from jarl.modules import MLP
from jarl.modules.encoder import LinearEncoder
from jarl.modules.operator import Critic
from jarl.modules.policy import MultiCategoricalPolicy
from jarl.runtime import OnPolicySchedule, Trainer
from jarl.sample import RolloutMinibatches
from jarl.store import RolloutBuffer
from jarl.transform import GAE

from physics_utils import forward_up_to_quat
from tracker_checkpoint import PeriodicCheckpoint


POSITION_SCALE     = (4108.0, 6000.0, 2076.0)
BALL_MAX_SPEED     = 6000.0
BALL_MAX_ANG_SPEED = 6.0
CAR_MAX_SPEED      = 2300.0
CAR_MAX_ANG_SPEED  = 5.5
BOOST_MAX          = 100.0
REPLAY_STATE_SIZE  = 137
BASE_STATE_SIZE    = 51


class ExpertGoalStates:

    _n_demos: int
    _min_len: int
    _windows: th.Tensor
    _demo_id: th.Tensor
    _replays: th.Tensor
    _offsets: th.Tensor
    _cursors: th.Tensor
    _times:   th.Tensor

    def __init__(
        self,
        replay_dir: str,
        n_env:      int,
        windows:    List[int] = [1, 2, 4, 8],
        obs_limit:  int | None = None,
        n_cars:     int = 2,
        device:     str | th.device = "cuda:0",
    ) -> None:

        self.n_cars = n_cars
        self.device = device

        replays, total = [], 0

        self._min_len = max(windows)

        for path in Path(replay_dir).glob("*.npy"):
            demos = self._filter(np.load(path, mmap_mode="r"))
            replays.extend(demos)

            total += sum((len(d) for d in demos))

            if obs_limit is not None and total >= obs_limit:
                break

        lengths = th.tensor([len(r) for r in replays], device=device)

        self._n_demos = len(replays)
        self._demo_id = th.zeros(n_env, device=device).long()
        self._windows = th.tensor(windows).to(device)[None, :]
        self._replays = th.concat(replays).to(device)
        self._offsets = th.cat((th.zeros(1, device=device, dtype=th.long), lengths.cumsum(0)))
        self._cursors = th.zeros(n_env, device=device).long()
        self._times   = th.zeros_like(self._cursors)

    def _filter(self, demo: np.ndarray) -> List[np.ndarray]:
        observation = demo[:, :-2]
        valid = np.append(~demo[:, -1].astype(bool), False)

        demos, start = [], 0

        for i, valid in enumerate(valid):
            if valid:
                continue

            if i - start >= self._min_len:
                demos.append(
                    th.from_numpy(observation[start:i])
                )

            start = i + 1

        return demos

    def reset(self, mask: th.Tensor) -> TensorBatch:
        n_resets = mask.sum().item()
        demo_id = th.randint(self._n_demos, (n_resets,), device=self.device)
        self._demo_id[mask] = demo_id

        starts = self._offsets[demo_id]
        ends = self._offsets[demo_id + 1]

        u = th.rand(n_resets, device=self.device)
        self._cursors[mask] = starts + (u * (ends - starts - self._min_len)).long()
        self._times[mask] = 0

        return TensorBatch({
            "observation": CARLObservation.from_tensor(
                self._replays[self._cursors[mask]],
                self.n_cars
            )
        })

    def next_goals(self, obs: CARLObservation) -> Tuple[TensorBatch, np.ndarray]:
        goal_idx = self._cursors[:, None] + self._windows
        goals = (self._replays[goal_idx, :31] - obs[:, None, :31]).flatten(-2)
        reset = (self._cursors + self._min_len) >= self._offsets[self._demo_id + 1]

        self._times += 1
        self._cursors += 1

        print(obs.shape)
        print(goals.shape)

        return (th.cat((obs, goals), dim=-1), reset)

In [75]:
goals = ExpertGoalStates(
    "ballchasing_replays/parsed_replays",
    n_env=1024,
    obs_limit=1_000_000
)

In [76]:
obs = goals.reset(th.ones(1024, dtype=th.bool, device="cuda")).data.get("observation")[0:]

In [77]:
for v in goals.next_goals(obs)[1]:
    print(v)

torch.Size([1024, 137])
torch.Size([1024, 124])
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')
tensor(True, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='c